# Tuba v4 Visualization Gallery

**A complete showcase of every visualization and export format.**

Tuba v4 produces piping stress results ? but results are only useful if they reach the right people
in the right format. This notebook demonstrates all **7 output channels** available out of the box:

| # | Format | Function | Output |
|---|--------|----------|--------|
| 1 | Interactive PyVista | `plots.plot_deformed_stress()` | Inline 3D in Jupyter |
| 2 | Standalone HTML | `export_html()` | Self-contained `.html` with vtk.js |
| 3 | Semantic Scene Bundle | `write_scene_bundle()` | JSON directory for web viewer |
| 4 | Static HTML Report | `write_static_report()` | Shareable report folder |
| 5 | PLY Export | `export_ply()` | Blender with vertex colors |
| 6 | glTF Export | `export_gltf()` | Universal 3D interchange |
| 7 | Blender Python Script | `export_blender_script()` | Full Blender scene generator |

We'll build a single piping model, load Code_Aster solver artifacts, then run the verified result data through every output.

In [ ]:
# ?? Setup: Repo root & imports ??????????????????????????????????????
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pyvista as pv
# Defaults to zoomable 'client' locally; set TUBA_NOTEBOOK_BACKEND=static for nbconvert/CI.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

from tuba import Model
from tuba.analysis.code_aster_notebook import load_or_run_code_aster_results

In [ ]:
# ── Build the demo model: L-shaped pipe with vertical riser ────────
#
# Geometry:
#   [0,0,0] ── 3 m straight ── 90° bend (XY) ── 2 m straight
#           ── 90° bend (XZ) ── 2 m vertical riser ── anchor
#
# Supports: anchors at both ends, guide at first bend, rest at second bend.

model = Model("VizGalleryDemo", standard="ASME_B31.3")

model.add_material(
    "Steel",
    E=2.1e11, nu=0.3, rho=7850.0, alpha=1.2e-5,
    allowable_stress={20.0: 137e6, 150.0: 127e6},
)

model.add_pipe_section("DN100", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

model.define_load_case(
    "Operating",
    gravity=True,
    pressure=1.5e6,
    temperature=150.0,
    ref_temperature=20.0,
)

with model.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(3.0)
    b.add_support(type="guide")
    b.bend(radius=0.3, angle=90, plane="XY")
    b.run(2.0)
    b.add_support(type="rest")
    b.bend(radius=0.3, angle=90, plane="XZ")
    b.run(2.0)
    b.end(support="anchor")

model.validate()
print(f"Model '{model.project_name}': {len(model.nodes)} nodes, {len(model.elements)} elements")
print(f"Sections: {list(model.sections.keys())}")
print(f"Load cases: {list(model.load_cases.keys())}")

In [ ]:
# ?? Load Code_Aster solver results ?????????????????????????????????
RUN_CODE_ASTER = True
CODE_ASTER_EXEC_METHOD = "wsl"  # Use "docker" if your Code_Aster installation is containerized.
CODE_ASTER_DOCKER_IMAGE = None
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "viz_gallery_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_EXEC_METHOD,
    docker_image=CODE_ASTER_DOCKER_IMAGE,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

if code_aster_run.ran_solver:
    print("Code_Aster solver executed because result tables were missing.")
print(f"Loaded Code_Aster results from: {CODE_ASTER_WORK_DIR.resolve()}")
print(f"Node result count: {len(results.node_results)}")
print(f"Element result count: {len(results.element_results)}")

---

## 1. Interactive PyVista Rendering

The primary notebook visualization. PyVista renders an inline 3D widget with full
**orbit / pan / zoom** controls directly in the Jupyter cell output.

Available plot functions in `tuba.visualizer.plots`:

| Function | What it shows |
|----------|---------------|
| `plot_deformed` | Deformed shape only |
| `plot_stress` | Un-deformed stress contour |
| `plot_deformed_stress` | **Deformed shape + stress colours** (primary view) |
| `plot_displacement_vectors` | Displacement arrows at each node |
| `plot_reactions` | Reaction force arrows at supports |
| `plot_temperature` | Temperature field contour |

In [ ]:
# ── 1. Interactive PyVista — Deformed Stress View ──────────────────
from tuba.visualizer import plots

plots.plot_deformed_stress(results, deform_scale=50.0, model=model)

---

## 2. Standalone HTML Export (vtk.js)

Generates a **self-contained HTML file** with an embedded 3D viewer powered by vtk.js.
Share with anyone — **no Python installation required**. The recipient just opens the
`.html` file in any modern browser and gets full interactive 3D.

In [ ]:
# ── 2. Standalone HTML Export ──────────────────────────────────────
from tuba.visualizer.export import export_html

export_html(results, "viz_gallery_deformed.html", model=model)
print("✓ Open viz_gallery_deformed.html in your browser for interactive 3D review.")

---

## 3. Semantic Scene Bundle (Web Viewer)

A directory of **structured JSON files** describing every scene object, overlay, issue,
and route review. This is the native data format consumed by the **Tuba web viewer**
(Three.js front-end).

Bundle structure:
```
viz_gallery_bundle/
├── scene.json              ← master manifest
├── metadata/
│   ├── objects.json         ← all scene objects
│   ├── overlays.json        ← stress overlays, etc.
│   └── issues.json          ← flagged issues
└── geometry/
    └── *.json               ← geometry assets
```

In [ ]:
# ?? 3. Semantic Scene Bundle ??????????????????????????????????????
from tuba.analysis import create_operating_geometry_state, create_visual_deformed_geometry_state
from tuba.visualization import (
    build_visualization_scene,
    SceneBuildOptions,
    write_scene_bundle,
)

operating_state = create_operating_geometry_state(model=model, result_state=code_aster_artifact.result_state)
visual_state = create_visual_deformed_geometry_state(
    model=model,
    result_state=code_aster_artifact.result_state,
    visual_scale=50.0,
)
analysis_meshes = [code_aster_artifact.analysis_mesh] if code_aster_artifact.analysis_mesh is not None else []
scene = build_visualization_scene(
    model,
    options=SceneBuildOptions(),
    analysis_meshes=analysis_meshes,
    result_states=[code_aster_artifact.result_state],
    geometry_states=[operating_state, visual_state],
)
bundle = write_scene_bundle(scene, "viz_gallery_bundle")

print(f"Scene bundle written to: {bundle.root}")
print(f"  scene.json : {bundle.scene_path}")
print(f"  metadata/  : {bundle.metadata_dir}")
print(f"  geometry/  : {bundle.geometry_dir}")

---

## 4. Static HTML Report

A **shareable report folder** with an `index.html` page, issue summary, and all scene
data embedded. Ideal for engineering review hand-off — zip the folder, email it, done.
The recipient gets a fully navigable report without any server.

In [ ]:
# ── 4. Static HTML Report ────────────────────────────────────────
from tuba.visualization import write_static_report

report = write_static_report(
    scene,
    "viz_gallery_report",
    title="Tuba v4 — Stress Analysis Report",
)

print(f"Report index        : {report.index_path}")
print(f"Issue summary       : {report.issue_summary_path}")
print(f"Report manifest     : {report.manifest_path}")

---

## 5. PLY Export (Blender with Vertex Colors)

Exports a 3D mesh in **PLY format** with per-vertex RGB stress colours baked in.
Import directly into Blender — the Von Mises stress gradient is visible immediately
via the **Vertex Color** attribute (no material setup required).

Workflow: `export_ply()` → Blender → *File → Import → PLY* → switch to Material Preview.

In [ ]:
# ── 5. PLY Export ────────────────────────────────────────────────
from tuba.visualizer.export import export_ply

export_ply(results, "viz_gallery_stress.ply", model=model, cmap="turbo")
print("✓ PLY file with Von Mises vertex colors exported.")

---

## 6. glTF Export (Universal 3D)

**glTF is the "JPEG of 3D"** — the most widely supported 3D interchange format.
Import into any 3D viewer, CAD tool, game engine, or web application.

Supported viewers: VS Code (glTF Tools), Windows 3D Viewer, Three.js, Babylon.js,
Blender, Unity, Unreal Engine, and more.

In [ ]:
# ── 6. glTF Export ───────────────────────────────────────────────
from tuba.visualizer.export import export_gltf

export_gltf(results, "viz_gallery_model.gltf", model=model)
print("✓ glTF model exported for universal 3D viewing.")

---

## 7. Blender Python Script

Generates a complete **Blender Python script** that recreates the pipe geometry
with stress-coloured materials from scratch. Run it headless or inside the Blender UI:

```bash
# Headless
blender --background --python viz_gallery_blender.py

# Inside Blender
# Scripting workspace → Open → viz_gallery_blender.py → Run Script
```

The script creates curve objects with bevel for each pipe element and assigns
per-vertex Von Mises stress as vertex colours.

In [ ]:
# ── 7. Blender Python Script ─────────────────────────────────────
from tuba.visualizer.export import export_blender_script

export_blender_script(results, "viz_gallery_blender.py", model=model)
print("✓ Blender script generated. Run inside Blender: File → Run Script")

---

## Summary

| Format | File Type | Use Case | Interactivity |
|--------|-----------|----------|---------------|
| **PyVista** | Jupyter widget | Day-to-day engineering review | Full 3D orbit/pan/zoom |
| **HTML Export** | `.html` | Share with non-Python colleagues | Full 3D in browser |
| **Scene Bundle** | JSON directory | Feed the Tuba web viewer | Programmable via Three.js |
| **Static Report** | HTML folder | Engineering review hand-off | Static + embedded scene |
| **PLY** | `.ply` | Blender rendering / animation | Vertex-color stress map |
| **glTF** | `.gltf` / `.glb` | Universal 3D interchange | Viewer-dependent |
| **Blender Script** | `.py` | Full Blender scene automation | Complete Blender control |

All formats start from the same Code_Aster `FEAResults` + `Model` objects. Choose the output
that matches your audience:

- **Yourself?** ? PyVista inline.
- **A colleague without Python?** ? HTML export or static report.
- **A web application?** ? Scene bundle or glTF.
- **A rendering artist?** ? PLY or Blender script.

---

*Next up: [05_compliance_checking.ipynb](./05_compliance_checking.ipynb) ? ASME B31.3 compliance evaluation.*